Evaluation Metrics in Machine Learning



Classification Metrics
Classification problems aim to predict discrete categories.

1. Accuracy
- used to evaluate the performacne of a classification model
- tells us the propertion of correct predictions made by th emodel out of all predictions 

Accuracy = No. of Correct Predictions / Total Number of Predictions 

Accuracy  =  ( 𝑇P + TN ) / (TP + TN + FP + FN)
	​
It answers: “What fraction did we classify correctly at a fixed threshold/decision rule?”

**When it’s reasonable to use accuracy**

Use it when all of these are approximately true:

	1. Class balance is roughly even (no extreme majority/minority class).
	2. Error costs are symmetric (FP and FN “hurt” similarly).
	3. Decision, not ranking is what matters (you must pick exactly one class).
	4. Distribution is stable (train/val/test prevalence similar; no big shift).
	5. No abstention/top-k requirement (you always predict exactly one label).

Typical fits: handwriting digit recognition with balanced digits, simple image classification benchmarks, quality checks where misses and false alarms are comparable.



When you should not rely on accuracy (pitfalls)

Class imbalance (accuracy paradox):
A “always-negative” model can get 99% accuracy if positives are 1%. You’ll miss every positive. Prefer PR-AUC, Recall@FPR, F1, or Balanced Accuracy here.

Asymmetric costs:
In fraud/medical screening, missing a positive (FN) is far worse than a false alarm (FP). Accuracy treats them equally → misleading threshold choices. Use a cost-sensitive metric, e.g., expected cost, or report recall at a target FPR.

Probability quality / calibration:
Accuracy ignores how confident you were. Overconfident wrong ≠ slightly wrong. Use Log Loss or Brier Score + calibration curves.

Ranking/retrieval/top-k tasks:
If you surface a shortlist (top-k) or care about ordering, accuracy is the wrong lens. Use NDCG@k, MAP, Recall@k, Hit@k.

Multi-label problems:
Exact-match accuracy is overly harsh; subset accuracy often near zero. Prefer micro/macro F1, Hamming loss, or label-ranking loss.

Changing base rates (dataset shift):
If positive prevalence at deployment differs from the test set, accuracy swings without the model changing. Track per-class metrics and monitor prevalence.

Small samples (high variance):
A single misclassification can move accuracy a lot. Always add confidence intervals (binomial/Wilson or bootstrap), and do cross-validation.

Group imbalance & fairness:
High overall accuracy can hide poor performance for a subgroup. Segment and report per-group accuracy/recall/precision, and fairness gaps (e.g., TPR parity).

Threshold gaming:
Maximizing accuracy picks the threshold where FP≈FN under equal costs; this is often not the business-optimal threshold.

Label noise caps achievable accuracy:
With noisy ground truth, the ceiling might be <100%. Track inter-annotator agreement (e.g., Cohen’s κ) and treat “perfect accuracy” claims skeptically.

Safer alternatives / companions

Balanced Accuracy = (TPR + TNR)/2 (mitigates imbalance).

F1 / Fβ (when both precision & recall matter; tune β to favor recall/precision).

ROC-AUC (ranking quality, cost-agnostic) and PR-AUC (rare positives).

MCC (Matthews Corr. Coef.)—robust single-number summary for binary with imbalance.

Cost-based decision metrics: expected cost at chosen threshold; Recall@FPR=x%.

Calibration: Log Loss, Brier, reliability diagrams if you use probabilities downstream.

Quick examples that fool accuracy

99% negatives, 1% positives:
Majority baseline (always predict negative): Accuracy = 99%, Recall = 0%, AUPRC = 0.01 (baseline) → terrible detector despite “great” accuracy.

Two subgroups:
Group A (80% of data): 99% accuracy; Group B (20%): 70% accuracy. Overall ≈93%, but Group B is harmed. Always slice.

Practical checklist (before you publish accuracy)

Report confusion matrix (+ precision, recall, specificity).

Add 95% CI for accuracy (Wilson or bootstrap).

Provide per-class and per-slice metrics.

If optimizing a threshold, justify it via costs or constraints (e.g., FPR≤1%).

If classes are imbalanced, include PR-AUC and Balanced Accuracy/MCC.

Minimal code: accuracy done right (with CIs & slices)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

# y_true: 1D array of true labels; y_pred: predicted labels (after threshold)
def wilson_interval(p, n, z=1.96):
    # 95% Wilson CI for a proportion
    denom = 1 + z**2/n
    center = (p + z**2/(2*n)) / denom
    margin = (z*np.sqrt((p*(1-p) + z**2/(4*n))/n)) / denom
    return center - margin, center + margin

def accuracy_report(y_true, y_pred, groups=None, labels=None):
    # Overall
    acc = accuracy_score(y_true, y_pred)
    lo, hi = wilson_interval(acc, len(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(f"Accuracy: {acc:.4f} (95% CI {lo:.4f}–{hi:.4f})")
    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)

    # Per-class precision/recall/F1
    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=0
    )
    for i, lab in enumerate(labels if labels is not None else np.unique(y_true)):
        print(f"Class {lab}: P={prec[i]:.3f} R={rec[i]:.3f} F1={f1[i]:.3f} (n={support[i]})")

    # Optional: per-group slices to avoid masking harms
    if groups is not None:
        groups = np.asarray(groups)
        for g in np.unique(groups):
            idx = (groups == g)
            a = accuracy_score(np.asarray(y_true)[idx], np.asarray(y_pred)[idx])
            lo, hi = wilson_interval(a, idx.sum())
            print(f"[Slice={g}] Accuracy: {a:.4f} (95% CI {lo:.4f}–{hi:.4f})")

# Example with imbalance
rng = np.random.default_rng(0)
y_true = (rng.random(5000) < 0.05).astype(int)          # 5% positives
# Bad baseline: predict all zeros
y_pred_bad = np.zeros_like(y_true)
accuracy_report(y_true, y_pred_bad, labels=[0,1])

# Better: pretend we have a thresholded model output
proba = rng.random(5000) * 0.5 + 0.25*y_true            # slightly higher for positives
tau = 0.5                                                # WARNING: maximizing accuracy may be misaligned
y_pred = (proba >= tau).astype(int)
accuracy_report(y_true, y_pred, labels=[0,1])


What to look for in the output

The “all-negative” baseline will show high accuracy but recall=0 for the positive class.

The second block illustrates why you must inspect per-class metrics even when overall accuracy rises only a little.


Accuracy is fine when classes are balanced and FP≈FN in cost.

Do not trust it alone with imbalance, asymmetric costs, ranking tasks, or probability-driven decisions.

Always pair accuracy with confusion matrix, per-class metrics, CIs, and at least one imbalance-aware metric (PR-AUC, F1, MCC, or Balanced Accuracy).